# Lab 6: Calculus and Gradient Descent

> **Course:** Data Science Foundations
> **Lesson:** Day 6 Lab, companion to `6-calculus-essentials.md`
> **Dataset:** `mpg` (seaborn), the same regression problem as Lab 5
> **Estimated time:** 2 hours

## Lab Objectives

- Compare forward and central finite differences, and see why $h$ can be *too* small
- Differentiate exponentials, logarithms and the **sigmoid**, and check each numerically
- Compute partial derivatives and verify a gradient with a **gradient check**
- Implement gradient descent in 1-D and explain exactly when it converges or diverges
- See gradient descent get stuck in a **local minimum** on a non-convex function
- Fit linear regression by gradient descent with a proper stopping rule, and link the maximum learning rate to linear algebra
- Implement **mini-batch stochastic gradient descent**

**How to work through this lab**

- Cells marked `# TODO` are yours to complete. Replace each `None` or comment with working code.
- Run every **Checkpoint** cell. If it prints `passed`, move on. If it raises an error, fix your code first.
- Each **Question** needs a short written answer in the "Your answer" cell below it.
- The dataset is different from the lesson on purpose. Adapt the lesson code; don't paste it.
- Worked solutions are in the `solutions/` folder. Attempt every part yourself before you open them: the struggle is where the learning happens.

## Part 0: Setup

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler

sns.set_theme(style="whitegrid")
rng = np.random.default_rng(42)

## Part 1: How Small Should $h$ Be?

The lesson defines the derivative with a **forward** difference $\frac{f(x+h) - f(x)}{h}$ but its code uses a **central** difference $\frac{f(x+h) - f(x-h)}{2h}$. Find out why.

In [ ]:
def forward_diff(f, x, h):
    # TODO
    return None


def central_diff(f, x, h):
    # TODO
    return None


# TODO 1.2: for f = np.sin at x = 1 (exact derivative: np.cos(1)), compute the absolute error of
# both methods for h = 1e-1, 1e-2, ..., 1e-12, and plot error vs h on log-log axes

**Question 1.** Describe the two curves. Why does the error go *up* again when $h$ becomes very small? What $h$ would you use for each method?

**Your answer:**

*Write your answer here.*

## Part 2: Exponentials, Logarithms and the Sigmoid

The lesson's rule table has no $e^x$ or $\ln x$, yet these appear in almost every ML loss function. Add them:

| Function | Derivative |
|---|---|
| $e^x$ | $e^x$ |
| $\ln x$ | $1/x$ |
| $e^{g(x)}$ (chain rule) | $g'(x)\,e^{g(x)}$ |
| $\sigma(x) = \frac{1}{1 + e^{-x}}$ (sigmoid, used in logistic regression) | $\sigma(x)\,(1 - \sigma(x))$ |

For each function below, work out the derivative **by hand**, code it, and let the checkpoint compare it with `central_diff`.

In [ ]:
def sigmoid(x):
    return 1 / (1 + np.exp(-x))


problems = {
    # name: (f, your hand-derived derivative)
    "3x^4 - 2x + 5": (lambda x: 3 * x**4 - 2 * x + 5, lambda x: None),   # TODO
    "e^(2x)":        (lambda x: np.exp(2 * x),        lambda x: None),   # TODO
    "ln(x^2 + 1)":   (lambda x: np.log(x**2 + 1),     lambda x: None),   # TODO
    "x e^(-x)":      (lambda x: x * np.exp(-x),       lambda x: None),   # TODO (product rule)
    "sigmoid(x)":    (sigmoid,                        lambda x: None),   # TODO
}

In [ ]:
# Checkpoint: run this cell. If it prints 'passed', move on.
x_test = np.linspace(-2, 2, 41)
for name, (f, df) in problems.items():
    numeric = central_diff(f, x_test, 1e-5)
    assert df(x_test) is not None, f"{name}: derivative not implemented"
    rel_err = np.max(np.abs(df(x_test) - numeric) / (1 + np.abs(numeric)))
    assert rel_err < 1e-6, f"{name}: max relative error {rel_err:.2e}"
    print(f"{name:<15s} OK (max rel. error {rel_err:.1e})")
print("passed")

In [ ]:
# TODO 2.2: plot sigmoid(x) and its derivative on x in [-8, 8]. Where is the derivative largest, and what is its value?

**Your answer:**

*Write your answer here.*

## Part 3: Partial Derivatives and the Gradient Check

$$f(x, y) = x^2 y + 3y^3 - 2x$$

When you code a gradient by hand, the standard safety net is a **gradient check**: compare it with a numerical gradient, one coordinate at a time.

In [ ]:
def f(p):
    x, y = p
    return x**2 * y + 3 * y**3 - 2 * x


def grad_f(p):
    # TODO: return np.array([df/dx, df/dy]), derived by hand
    return None


def numerical_gradient(func, p, h=1e-5):
    # TODO: central difference along each coordinate. Works for any length of p.
    # For coordinate i: nudge p[i] by +h and -h (copy p first!) and apply the central difference formula.
    return None

In [ ]:
# Checkpoint: run this cell. If it prints 'passed', move on.
for p in rng.normal(size=(5, 2)) * 3:
    assert np.allclose(grad_f(p), numerical_gradient(f, p), rtol=1e-5, atol=1e-6), p
assert np.allclose(grad_f([1, 2]), [2, 37])
print("passed")

In [ ]:
# TODO 3.2: for the bowl g(x, y) = (x - 1)^2 + 2 (y + 0.5)^2, draw a contour plot on [-2, 4] x [-3, 2]
# and overlay the NEGATIVE gradient as arrows (plt.quiver) on a coarse grid. Use ax.set_aspect("equal").

**Your answer:**

*Write your answer here.*

## Part 4: Gradient Descent in One Dimension

$$f(x) = (x - 3)^2 + 2, \qquad f'(x) = 2(x - 3)$$

In [ ]:
def f1(x):
    return (x - 3) ** 2 + 2


def df1(x):
    return 2 * (x - 3)


def gradient_descent_1d(df, x0, lr, n_iters):
    # TODO: return a NumPy array of all positions [x0, x1, ..., x_n_iters]
    return None


# TODO 4.2: from x0 = -2, run 30 iterations for lr in [0.01, 0.1, 0.5, 0.9, 1.05].
# Plot x_t against t for each learning rate on one figure (use ax.set_ylim(-10, 16)).

In [ ]:
# Checkpoint: run this cell. If it prints 'passed', move on.
path = gradient_descent_1d(df1, -2.0, 0.1, 100)
assert len(path) == 101 and abs(path[-1] - 3) < 1e-6
print("passed")

**Question 4.** Substitute the update into itself: show that $x_{t+1} - 3 = (1 - 2\alpha)(x_t - 3)$. Use it to explain every curve you plotted, and find the exact range of learning rates for which gradient descent converges.

**Your answer:**

*Write your answer here.*

## Part 5: Non-Convex Functions and Local Minima

Linear regression's loss is a convex bowl, so gradient descent always finds the global minimum. Most ML losses are not convex:

$$f(x) = x^4 - 3x^2 + x, \qquad f'(x) = 4x^3 - 6x + 1$$

In [ ]:
def f2(x):
    return x**4 - 3 * x**2 + x


def df2(x):
    return 4 * x**3 - 6 * x + 1


# TODO: run gradient_descent_1d(df2, x0, lr=0.01, n_iters=500) from x0 in [-2, -0.5, 0.5, 2].
# Plot f2 on [-2.2, 2.2] and mark each start and end point. Print where each run ends and f2 there.

**Your answer:**

*Write your answer here.*

## Part 6: Linear Regression by Gradient Descent, with a Stopping Rule

The lesson runs a fixed 1,000 iterations. A better loop stops when the gradient is tiny: $\|\nabla L\| < \text{tol}$. Same data preparation as Lab 5 (split first, then scale).

In [ ]:
cars = sns.load_dataset("mpg").dropna().reset_index(drop=True)
predictors = ["cylinders", "displacement", "horsepower", "weight", "acceleration", "model_year"]
X = cars[predictors].to_numpy(dtype=float)
y = cars["mpg"].to_numpy()

X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)
scaler = StandardScaler().fit(X_train)
Xd_train = np.hstack([np.ones((len(X_train), 1)), scaler.transform(X_train)])
Xd_test = np.hstack([np.ones((len(X_test), 1)), scaler.transform(X_test)])
w_exact = np.linalg.solve(Xd_train.T @ Xd_train, Xd_train.T @ y_train)
print("Normal-equation weights:", w_exact.round(3))

In [ ]:
def gd_linear_regression(X, y, lr, tol=1e-6, max_iters=200_000):
    # TODO: start from w = 0. Each iteration:
    #   grad = -(2/m) X^T (y - X w); record the MSE in `losses`;
    #   stop if ||grad|| < tol, or if the loss becomes nan/inf (diverged);
    #   w = w - lr * grad
    # return w, losses (list), n_iters_used
    return None

In [ ]:
# Checkpoint: run this cell. If it prints 'passed', move on.
w_gd, losses, n_used = gd_linear_regression(Xd_train, y_train, lr=0.05)
print(f"converged in {n_used} iterations; max |w_gd - w_exact| = {np.abs(w_gd - w_exact).max():.2e}")
assert np.allclose(w_gd, w_exact, atol=1e-4)
print("passed")

### 6.2 The maximum learning rate comes from linear algebra

The MSE loss has a constant second derivative (its **Hessian**) $H = \frac{2}{m} X^TX$. Generalising Part 4, gradient descent converges exactly when $\alpha < 2 / \lambda_{\max}$, where $\lambda_{\max}$ is the largest **eigenvalue** of $H$ (the curvature along the steepest direction of the bowl). The *smallest* eigenvalue sets how slowly the flattest direction converges.

In [ ]:
# TODO 6.2: compute the eigenvalues of H with np.linalg.eigvalsh, then lr_max = 2 / largest eigenvalue
lr_max = None

# TODO 6.3: run gd_linear_regression for lr in [0.01, 0.05, 0.1, 0.9 * lr_max, 1.02 * lr_max]
# (max_iters=20_000). Report iterations used, final loss and whether it converged. Plot all loss curves (log y-axis).

**Question 6.** What is `lr_max` for this problem? What happens just below and just above it? Why does even the best learning rate still need many iterations here? (Hint: look at the condition number and recall Lab 5's collinear features.)

**Your answer:**

*Write your answer here.*

## Part 7: Mini-Batch Stochastic Gradient Descent (extension)

Full-batch GD computes the gradient over all $m$ rows for every step. **Mini-batch SGD** shuffles the data each **epoch** (one pass through it) and takes one step per batch of $b$ rows, using that batch's gradient as a cheap, noisy estimate. The lesson mentions SGD; here you build it.

In [ ]:
def minibatch_sgd(X, y, lr, batch_size, n_epochs, rng=rng):
    # TODO: w starts at 0. For each epoch:
    #   shuffle the row indices (rng.permutation), loop over them in chunks of batch_size,
    #   compute the gradient on that chunk only (divide by the chunk size), update w.
    #   After each epoch record the FULL training MSE in `epoch_losses`.
    # return w, epoch_losses
    return None


# TODO 7.2: run 40 epochs with lr = 0.01 and batch_size in [len(y_train), 32, 1].
# Plot the loss per epoch for each (log y-axis) and print the final loss vs the exact minimum loss.

**Your answer:**

*Write your answer here.*

## Part 8: Wrap-Up

In 4 to 5 sentences, connect this week: how do the gradient (calculus), $X^TX$ and its eigenvalues (linear algebra), and the MSE loss (statistics) work together when a model "learns"?

**Your answer:**

*Write your answer here.*

> Submit your completed notebook to the Kanban board under your name by end of day.